In [1]:
# 4. Design your function here! 

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df

In [2]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.
import ee

#ee.Authenticate()

ee.Initialize()

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

#Plumas = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")

# This is a dictionary that my code requires. You don't have to touch anything here for demo purposes
# (although it should work with anything, in theory. Feel free to change it if you'd like).
# These parameters get stored and are used to generate the header information needed when we do the full
# run.

vector_path = r"C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\geometry\Plumas_National_forest.geojson"

"""
parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2018-05-08',
            'end_date': '2018-05-10',
            'vector_path': vector_path,
            'crs': 'EPSG:3310',
            'scale': 30,
            'map_function': prep_sr_l8
        }
"""

ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-05-08', '2018-05-10').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

dataset_params = {
    'geometry': vector_path,
    'crs': 'EPSG:3310',
    'scale': 30
}

In [ ]:
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_params=dataset_params,
    user_function=compute_ndvi,
    export_params={"flag": "GTiff", "output_folder": "rush123"},
    dask_mode="full"  # for single worker in local test mode
)

c:\anaconda3\envs\rredit\lib\site-packages\distributed\node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 55852 instead
  warnings.warn(
c:\anaconda3\envs\rredit\lib\contextlib.py:142: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.07s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Dask dashboard is available at: http://127.0.0.1:55852/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:55853' processes=16 threads=16, memory=29.59 GiB>
Dask workers authenticated via to Earth Engine...
<xarray.Dataset> Size: 264MB
Dimensions:  (time: 2, X: 4741, Y: 3481)
Coordinates:
  * time     (time) datetime64[ns] 16B 2018-05-09T18:38:10.899000 2018-05-09T...
  * X        (X) float64 38kB -1.459e+05 -1.458e+05 ... -3.69e+03 -3.66e+03
  * Y        (Y) float64 28kB 1.516e+05 1.516e+05 ... 2.559e+05 2.56e+05
Data variables:
    SR_B4    (time, X, Y) float32 132MB dask.array<chunksize=(2, 512, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 132MB dask.array<chunksize=(2, 512, 256), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:        